# Reglas de inferencia resueltas con tablas de verdad

Cuaderno complementario al capítulo *2.3 Reglas de Inferencia y Deducción Natural*.

Un argumento con premisas $P_1,\dots,P_k$ y conclusión $C$ es **válido** si no existe ninguna asignación de valores de verdad que haga verdaderas *todas* las premisas y falsa la conclusión. Equivalentemente:

$$(P_1\land P_2\land\cdots\land P_k)\to C \text{ es una tautología.}$$

**Método para cada regla:**
1. Se listan las $2^n$ asignaciones de las variables proposicionales.
2. Se calculan las columnas de cada premisa y de la conclusión.
3. Se marcan las **filas críticas** (todas las premisas en V). En ellas la conclusión debe ser V.
4. Se comprueba que la columna final $(\bigwedge P_i)\to C$ sea V en todas las filas (tautología).

Reglas tratadas (sección 1 del capítulo): A. Modus Ponens · B. Modus Tollens · C. Silogismo Disyuntivo · D. Silogismo Hipotético · E. Dilema Constructivo · F. Simplificación · G. Adición · H. Conjunción. Al final se verifica el ejemplo de deducción de la sección 3 y se contrastan con dos **falacias** que *no* son reglas válidas.

In [1]:
import itertools
import pandas as pd

V = lambda b: 'V' if b else 'F'
impl = lambda a, b: (not a) or b

def argumento(nombre, variables, premisas, conclusion, mostrar=True):
    # premisas: lista de (etiqueta, función(**vars)->bool); conclusion: (etiqueta, función)
    filas = []
    for vals in itertools.product([True, False], repeat=len(variables)):
        env = dict(zip(variables, vals))
        ps = [f(**env) for _, f in premisas]
        c = conclusion[1](**env)
        todas = all(ps)
        fila = {v: V(env[v]) for v in variables}
        for (et, _), val in zip(premisas, ps):
            fila[et] = V(val)
        fila['∧ premisas'] = V(todas)
        fila[conclusion[0]] = V(c)
        fila['fila crítica'] = '★' if todas else ''
        fila['(∧ premisas) → conclusión'] = V(impl(todas, c))
        filas.append(fila)
    df = pd.DataFrame(filas)
    criticas = df[df['fila crítica'] == '★']
    valido = (df['(∧ premisas) → conclusión'] == 'V').all()
    if mostrar:
        print('=' * 72); print(nombre); print('=' * 72)
        sty = df.style.apply(lambda r: ['background-color:#fff3b0; color:#000' if r['fila crítica'] else '' for _ in r], axis=1)
        display(sty)
        print(f'Filas críticas (todas las premisas V): {len(criticas)} de {len(df)}')
        print('En todas ellas la conclusión es V   :', (criticas[conclusion[0]] == 'V').all())
        print('Condicional (∧ premisas) → conclusión es tautología:', V(valido))
        print('ARGUMENTO', 'VÁLIDO ✔' if valido else 'INVÁLIDO ✘', '(hay fila crítica con conclusión F)' if not valido else '')
    return valido, df

## A. Modus Ponendo Ponens (m.p.p.)

$$\frac{p\to q\quad\land\ p}{\therefore q}$$

In [2]:
ok, _ = argumento('A. Modus Ponendo Ponens (m.p.p.)', ['p', 'q'],
    premisas=[('p→q', lambda p,q: impl(p,q)), ('p', lambda p,q: p)],
    conclusion=('q', lambda p,q: q))
assert ok

A. Modus Ponendo Ponens (m.p.p.)


,p,q,p→q,∧ premisas,fila crítica,(∧ premisas) → conclusión
0,V,V,V,V,★,V
1,V,F,F,F,,V
2,F,V,V,F,,V
3,F,F,V,F,,V


Filas críticas (todas las premisas V): 1 de 4
En todas ellas la conclusión es V   : True
Condicional (∧ premisas) → conclusión es tautología: V
ARGUMENTO VÁLIDO ✔ 


**Lectura.** La única fila con $p\to q$ y $p$ ambas V es $p=V,q=V$: ahí $q=V$. Si $p$ es V y $q$ es F, la premisa $p\to q$ es F y la fila deja de ser crítica.

## B. Modus Tollendo Tollens (m.t.t.)

$$\frac{p\to q\quad\land\ \neg q}{\therefore \neg p}$$

In [3]:
ok, _ = argumento('B. Modus Tollendo Tollens (m.t.t.)', ['p', 'q'],
    premisas=[('p→q', lambda p,q: impl(p,q)), ('¬q', lambda p,q: not q)],
    conclusion=('¬p', lambda p,q: not p))
assert ok

B. Modus Tollendo Tollens (m.t.t.)


,p,q,p→q,¬q,∧ premisas,¬p,fila crítica,(∧ premisas) → conclusión
0,V,V,V,F,F,F,,V
1,V,F,F,V,F,F,,V
2,F,V,V,F,F,V,,V
3,F,F,V,V,V,V,★,V


Filas críticas (todas las premisas V): 1 de 4
En todas ellas la conclusión es V   : True
Condicional (∧ premisas) → conclusión es tautología: V
ARGUMENTO VÁLIDO ✔ 


**Lectura.** Con $\neg q$ verdadera ($q=F$) y $p\to q$ verdadera, sólo puede ser $p=F$, o sea $\neg p=V$.

## C. Modus Tollendo Ponens (m.t.p. / Silogismo Disyuntivo)

$$\frac{p\lor q\quad\land\ \neg p}{\therefore q}\qquad\text{o bien}\qquad\frac{p\lor q\quad\land\ \neg q}{\therefore p}$$

La regla tiene **dos formas**; se verifica cada una.

In [4]:
ok1, _ = argumento('C1. Silogismo Disyuntivo: p∨q, ¬p ∴ q', ['p','q'],
    premisas=[('p∨q', lambda p,q: p or q), ('¬p', lambda p,q: not p)],
    conclusion=('q', lambda p,q: q))
ok2, _ = argumento('C2. Silogismo Disyuntivo: p∨q, ¬q ∴ p', ['p','q'],
    premisas=[('p∨q', lambda p,q: p or q), ('¬q', lambda p,q: not q)],
    conclusion=('p', lambda p,q: p))
assert ok1 and ok2

C1. Silogismo Disyuntivo: p∨q, ¬p ∴ q


,p,q,p∨q,¬p,∧ premisas,fila crítica,(∧ premisas) → conclusión
0,V,V,V,F,F,,V
1,V,F,V,F,F,,V
2,F,V,V,V,V,★,V
3,F,F,F,V,F,,V


Filas críticas (todas las premisas V): 1 de 4
En todas ellas la conclusión es V   : True
Condicional (∧ premisas) → conclusión es tautología: V
ARGUMENTO VÁLIDO ✔ 
C2. Silogismo Disyuntivo: p∨q, ¬q ∴ p


,p,q,p∨q,¬q,∧ premisas,fila crítica,(∧ premisas) → conclusión
0,V,V,V,F,F,,V
1,V,F,V,V,V,★,V
2,F,V,V,F,F,,V
3,F,F,F,V,F,,V


Filas críticas (todas las premisas V): 1 de 4
En todas ellas la conclusión es V   : True
Condicional (∧ premisas) → conclusión es tautología: V
ARGUMENTO VÁLIDO ✔ 


## D. Silogismo Hipotético

$$\frac{p\to q\quad\land\ q\to r}{\therefore p\to r}$$

In [5]:
ok, _ = argumento('D. Silogismo Hipotético', ['p', 'q', 'r'],
    premisas=[('p→q', lambda p,q,r: impl(p,q)), ('q→r', lambda p,q,r: impl(q,r))],
    conclusion=('p→r', lambda p,q,r: impl(p,r)))
assert ok

D. Silogismo Hipotético


,p,q,r,p→q,q→r,∧ premisas,p→r,fila crítica,(∧ premisas) → conclusión
0,V,V,V,V,V,V,V,★,V
1,V,V,F,V,F,F,F,,V
2,V,F,V,F,V,F,V,,V
3,V,F,F,F,V,F,F,,V
4,F,V,V,V,V,V,V,★,V
5,F,V,F,V,F,F,V,,V
6,F,F,V,V,V,V,V,★,V
7,F,F,F,V,V,V,V,★,V


Filas críticas (todas las premisas V): 4 de 8
En todas ellas la conclusión es V   : True
Condicional (∧ premisas) → conclusión es tautología: V
ARGUMENTO VÁLIDO ✔ 


**Lectura.** Con tres variables hay $2^3=8$ filas. Las filas críticas son las que satisfacen $p\to q$ y $q\to r$; en todas $p\to r$ resulta V (transitividad).

## E. Dilema Constructivo

$$\frac{(p\to q)\land(r\to s)\quad\land\ (p\lor r)}{\therefore q\lor s}$$

In [6]:
ok, _ = argumento('E. Dilema Constructivo', ['p', 'q', 'r', 's'],
    premisas=[('(p→q)∧(r→s)', lambda p,q,r,s: impl(p,q) and impl(r,s)), ('p∨r', lambda p,q,r,s: p or r)],
    conclusion=('q∨s', lambda p,q,r,s: q or s))
assert ok

E. Dilema Constructivo


,p,q,r,s,(p→q)∧(r→s),p∨r,∧ premisas,q∨s,fila crítica,(∧ premisas) → conclusión
0,V,V,V,V,V,V,V,V,★,V
1,V,V,V,F,F,V,F,V,,V
2,V,V,F,V,V,V,V,V,★,V
3,V,V,F,F,V,V,V,V,★,V
4,V,F,V,V,F,V,F,V,,V
5,V,F,V,F,F,V,F,F,,V
6,V,F,F,V,F,V,F,V,,V
7,V,F,F,F,F,V,F,F,,V
8,F,V,V,V,V,V,V,V,★,V
9,F,V,V,F,F,V,F,V,,V


Filas críticas (todas las premisas V): 5 de 16
En todas ellas la conclusión es V   : True
Condicional (∧ premisas) → conclusión es tautología: V
ARGUMENTO VÁLIDO ✔ 


**Lectura.** $2^4=16$ filas. Si $p\lor r$ es V, al menos uno de $p,r$ es V; su implicación correspondiente fuerza $q$ o $s$ en V.

## F. Ley de Simplificación Conjuntiva

$$\frac{p\land q}{\therefore p}\qquad\text{y}\qquad\frac{p\land q}{\therefore q}$$

In [7]:
ok1, _ = argumento('F1. Simplificación: p∧q ∴ p', ['p','q'],
    premisas=[('p∧q', lambda p,q: p and q)], conclusion=('p', lambda p,q: p))
ok2, _ = argumento('F2. Simplificación: p∧q ∴ q', ['p','q'],
    premisas=[('p∧q', lambda p,q: p and q)], conclusion=('q', lambda p,q: q))
assert ok1 and ok2

F1. Simplificación: p∧q ∴ p


,p,q,p∧q,∧ premisas,fila crítica,(∧ premisas) → conclusión
0,V,V,V,V,★,V
1,V,F,F,F,,V
2,F,V,F,F,,V
3,F,F,F,F,,V


Filas críticas (todas las premisas V): 1 de 4
En todas ellas la conclusión es V   : True
Condicional (∧ premisas) → conclusión es tautología: V
ARGUMENTO VÁLIDO ✔ 
F2. Simplificación: p∧q ∴ q


,p,q,p∧q,∧ premisas,fila crítica,(∧ premisas) → conclusión
0,V,V,V,V,★,V
1,V,F,F,F,,V
2,F,V,F,F,,V
3,F,F,F,F,,V


Filas críticas (todas las premisas V): 1 de 4
En todas ellas la conclusión es V   : True
Condicional (∧ premisas) → conclusión es tautología: V
ARGUMENTO VÁLIDO ✔ 


## G. Ley de Adición Disyuntiva

$$\frac{p}{\therefore p\lor q}$$

In [8]:
ok, _ = argumento('G. Ley de Adición Disyuntiva', ['p', 'q'],
    premisas=[('p', lambda p,q: p)],
    conclusion=('p∨q', lambda p,q: p or q))
assert ok

G. Ley de Adición Disyuntiva


,p,q,∧ premisas,p∨q,fila crítica,(∧ premisas) → conclusión
0,V,V,V,V,★,V
1,V,F,V,V,★,V
2,F,V,F,V,,V
3,F,F,F,F,,V


Filas críticas (todas las premisas V): 2 de 4
En todas ellas la conclusión es V   : True
Condicional (∧ premisas) → conclusión es tautología: V
ARGUMENTO VÁLIDO ✔ 


**Lectura.** $q$ es *arbitraria*: aunque $q$ sea F, $p\lor q$ es V porque $p$ es V. Nótese que $q$ aparece sólo en la conclusión.

## H. Ley de Conjunción (introducción del ∧)

$$\frac{p\quad\land\ q}{\therefore p\land q}$$

In [9]:
ok, _ = argumento('H. Ley de Conjunción (introducción del ∧)', ['p', 'q'],
    premisas=[('p', lambda p,q: p), ('q', lambda p,q: q)],
    conclusion=('p∧q', lambda p,q: p and q))
assert ok

H. Ley de Conjunción (introducción del ∧)


,p,q,∧ premisas,p∧q,fila crítica,(∧ premisas) → conclusión
0,V,V,V,V,★,V
1,V,F,F,F,,V
2,F,V,F,F,,V
3,F,F,F,F,,V


Filas críticas (todas las premisas V): 1 de 4
En todas ellas la conclusión es V   : True
Condicional (∧ premisas) → conclusión es tautología: V
ARGUMENTO VÁLIDO ✔ 


**Lectura.** Sólo hay una fila crítica ($p=q=V$) y ahí $p\land q=V$.

## Resumen de validez

Se vuelve a evaluar cada regla y se reporta una tabla única (con el número de filas críticas y de filas totales).

In [10]:
reglas = {
 'A. Modus Ponens              p→q, p ⊢ q':        (['p','q'],   [lambda p,q: impl(p,q), lambda p,q: p],            lambda p,q: q),
 'B. Modus Tollens             p→q, ¬q ⊢ ¬p':      (['p','q'],   [lambda p,q: impl(p,q), lambda p,q: not q],        lambda p,q: not p),
 'C1. Silogismo Disyuntivo     p∨q, ¬p ⊢ q':       (['p','q'],   [lambda p,q: p or q, lambda p,q: not p],           lambda p,q: q),
 'C2. Silogismo Disyuntivo     p∨q, ¬q ⊢ p':       (['p','q'],   [lambda p,q: p or q, lambda p,q: not q],          lambda p,q: p),
 'D. Silogismo Hipotético      p→q, q→r ⊢ p→r':    (['p','q','r'], [lambda p,q,r: impl(p,q), lambda p,q,r: impl(q,r)], lambda p,q,r: impl(p,r)),
 'E. Dilema Constructivo       (p→q)∧(r→s), p∨r ⊢ q∨s': (['p','q','r','s'],
        [lambda p,q,r,s: impl(p,q) and impl(r,s), lambda p,q,r,s: p or r], lambda p,q,r,s: q or s),
 'F1. Simplificación           p∧q ⊢ p':           (['p','q'],   [lambda p,q: p and q],                            lambda p,q: p),
 'F2. Simplificación           p∧q ⊢ q':           (['p','q'],   [lambda p,q: p and q],                            lambda p,q: q),
 'G. Adición                   p ⊢ p∨q':           (['p','q'],   [lambda p,q: p],                                  lambda p,q: p or q),
 'H. Conjunción                p, q ⊢ p∧q':        (['p','q'],   [lambda p,q: p, lambda p,q: q],                   lambda p,q: p and q),
}
filas = []
for nombre, (vs, ps, c) in reglas.items():
    tot = crit = 0; valido = True
    for vals in itertools.product([True, False], repeat=len(vs)):
        env = dict(zip(vs, vals)); tot += 1
        if all(p(**env) for p in ps):
            crit += 1
            valido &= c(**env)
    filas.append({'Regla': nombre, 'Variables': len(vs), 'Filas': tot, 'Filas críticas': crit, 'Conclusión V en todas': V(valido)})
pd.DataFrame(filas).set_index('Regla')

,Variables,Filas,Filas críticas,Conclusión V en todas
Regla,,,,
"A. Modus Ponens p→q, p ⊢ q",2,4,1,V
"B. Modus Tollens p→q, ¬q ⊢ ¬p",2,4,1,V
"C1. Silogismo Disyuntivo p∨q, ¬p ⊢ q",2,4,1,V
"C2. Silogismo Disyuntivo p∨q, ¬q ⊢ p",2,4,1,V
"D. Silogismo Hipotético p→q, q→r ⊢ p→r",3,8,4,V
"E. Dilema Constructivo (p→q)∧(r→s), p∨r ⊢ q∨s",4,16,5,V
F1. Simplificación p∧q ⊢ p,2,4,1,V
F2. Simplificación p∧q ⊢ q,2,4,1,V
G. Adición p ⊢ p∨q,2,4,2,V


## Contraste: dos falacias que *parecen* reglas

La tabla de verdad también sirve para descartar razonamientos inválidos. Aquí aparece una fila crítica con conclusión F (**contraejemplo**).

- **Afirmación del consecuente:** $p\to q,\ q\ \therefore\ p$
- **Negación del antecedente:** $p\to q,\ \neg p\ \therefore\ \neg q$

In [11]:
ok1, _ = argumento('Falacia: afirmación del consecuente  (p→q, q ∴ p)', ['p','q'],
    premisas=[('p→q', lambda p,q: impl(p,q)), ('q', lambda p,q: q)], conclusion=('p', lambda p,q: p))
ok2, _ = argumento('Falacia: negación del antecedente  (p→q, ¬p ∴ ¬q)', ['p','q'],
    premisas=[('p→q', lambda p,q: impl(p,q)), ('¬p', lambda p,q: not p)], conclusion=('¬q', lambda p,q: not q))
assert not ok1 and not ok2

Falacia: afirmación del consecuente  (p→q, q ∴ p)


,p,q,p→q,∧ premisas,fila crítica,(∧ premisas) → conclusión
0,V,V,V,V,★,V
1,V,F,F,F,,V
2,F,V,V,V,★,F
3,F,F,V,F,,V


Filas críticas (todas las premisas V): 2 de 4
En todas ellas la conclusión es V   : False
Condicional (∧ premisas) → conclusión es tautología: F
ARGUMENTO INVÁLIDO ✘ (hay fila crítica con conclusión F)
Falacia: negación del antecedente  (p→q, ¬p ∴ ¬q)


,p,q,p→q,¬p,∧ premisas,¬q,fila crítica,(∧ premisas) → conclusión
0,V,V,V,F,F,F,,V
1,V,F,F,F,F,V,,V
2,F,V,V,V,V,F,★,F
3,F,F,V,V,V,V,★,V


Filas críticas (todas las premisas V): 2 de 4
En todas ellas la conclusión es V   : False
Condicional (∧ premisas) → conclusión es tautología: F
ARGUMENTO INVÁLIDO ✘ (hay fila crítica con conclusión F)


En ambos casos el contraejemplo es $p=F,\ q=V$: las premisas son V y la conclusión es F. Por eso *no* son reglas de inferencia, a diferencia de m.p.p. y m.t.t.

## Verificación del ejemplo de la sección 3

Argumento: $p\to q,\ q\to\neg r,\ r\lor s,\ p\ \therefore\ s$ (4 variables, 16 filas).

La derivación del capítulo usa m.p.p. dos veces y m.t.p. una vez. Con tablas de verdad se comprueba **directamente** que el argumento completo es válido, y además que cada paso intermedio también lo es.

In [12]:
ok, df = argumento('Ejemplo sec. 3: p→q, q→¬r, r∨s, p ∴ s', ['p','q','r','s'],
    premisas=[('p→q', lambda p,q,r,s: impl(p,q)),
              ('q→¬r', lambda p,q,r,s: impl(q, not r)),
              ('r∨s', lambda p,q,r,s: r or s),
              ('p', lambda p,q,r,s: p)],
    conclusion=('s', lambda p,q,r,s: s))
assert ok

Ejemplo sec. 3: p→q, q→¬r, r∨s, p ∴ s


,p,q,r,s,p→q,q→¬r,r∨s,∧ premisas,fila crítica,(∧ premisas) → conclusión
0,V,V,V,V,V,F,V,F,,V
1,V,V,V,F,V,F,V,F,,V
2,V,V,F,V,V,V,V,V,★,V
3,V,V,F,F,V,V,F,F,,V
4,V,F,V,V,F,V,V,F,,V
5,V,F,V,F,F,V,V,F,,V
6,V,F,F,V,F,V,V,F,,V
7,V,F,F,F,F,V,F,F,,V
8,F,V,V,V,V,F,V,F,,V
9,F,V,V,F,V,F,V,F,,V


Filas críticas (todas las premisas V): 1 de 16
En todas ellas la conclusión es V   : True
Condicional (∧ premisas) → conclusión es tautología: V
ARGUMENTO VÁLIDO ✔ 


In [13]:
# Pasos intermedios de la derivación: cada uno es consecuencia de las premisas
todas = [lambda p,q,r,s: impl(p,q), lambda p,q,r,s: impl(q, not r), lambda p,q,r,s: r or s, lambda p,q,r,s: p]
pasos = {
 '(5) q     [m.p.p. (1),(4)]':  lambda p,q,r,s: q,
 '(6) ¬r    [m.p.p. (2),(5)]':  lambda p,q,r,s: not r,
 '(7) s     [m.t.p. (3),(6)]':  lambda p,q,r,s: s,
}
res = []
for nombre, f in pasos.items():
    ok, _ = argumento(nombre, ['p','q','r','s'], [(f'P{i+1}', g) for i, g in enumerate(todas)], (nombre, f), mostrar=False)
    res.append({'Paso': nombre, 'Consecuencia lógica de P1..P4': V(ok)})
pd.DataFrame(res).set_index('Paso')

,Consecuencia lógica de P1..P4
Paso,
"(5) q [m.p.p. (1),(4)]",V
"(6) ¬r [m.p.p. (2),(5)]",V
"(7) s [m.t.p. (3),(6)]",V


La única fila crítica del argumento es $p=q=V,\ r=F,\ s=V$, y en ella la conclusión $s$ es V. Coincide con el resultado de la derivación formal, que llega a $s$ en el paso (7).

> **Nota.** Las reglas de cuantificadores (E.U., G.U., E.E., G.E.) de la sección 2 del capítulo no se tratan aquí: no son esquemas proposicionales, sino reglas sobre predicados con restricciones sobre constantes (arbitraria/nueva); se evalúan sobre universos finitos como en el cuaderno de cuantificadores.